# Etapa A: aprendizaje de la normalidad (autoencoder secuencial)

Antes de intentar clasificar qué remitente es sospechoso, en este notebook entrenamos
un modelo que aprende exclusivamente qué es un **remitente normal**. Nuestra idea de
partida fue que un modelo entrenado solo con comportamiento normal aprendería una
representación comprimida de ese comportamiento, de modo que al presentarle una
secuencia que no se parece a nada que haya visto, fallaría en reproducirla. Esa
dificultad de reproducción, el error de reconstrucción, es el score de anomalía.

Esta etapa junta dos temas del curso. De la Semana 4, Encoder Decoder Autoencoders,
tomamos la idea central: comprimir a un cuello de botella y reconstruir desde ahí, usando
el error de reconstrucción como señal. De la Semana 3, Redes Neuronales Recurrentes y
LSTM, tomamos la pieza que maneja la longitud variable de los historiales, porque cada
remitente tiene entre 3 y 32 transacciones. Por eso el encoder y el decoder son GRU.

## Qué produce este notebook

- `artifacts/stage_a_model.pt`: el autoencoder completo (encoder + decoder).
- `artifacts/encoder.pt`: solo el encoder, que es lo que reutilizamos en la Etapa B
  por transfer learning.
- `artifacts/anomaly_threshold.json`: el umbral que elegimos, su justificación y las
  métricas de detección en validación y prueba.

## Contrato con la Etapa B

Hicimos que el encoder no solo comprima la secuencia a un vector, sino que también
exponga el estado oculto de la GRU en **cada transacción**. Ese detalle es el que nos
permite implementar atención sobre transacciones individuales en la Etapa B. Sin él no
tendríamos manera de explicar qué movimientos activaron una alerta.

Consumimos únicamente `artifacts/{train,val,test}.npz`, que ya generamos en
[`01_data_engineering.ipynb`](01_data_engineering.ipynb). No volvemos a tocar el CSV
original ni descargamos nada de Kaggle.

In [1]:
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/Vann06/Deep-Learning.git"
BRANCH = "Proyecto2"

if "google.colab" in sys.modules:
    if not Path("src").exists():
        subprocess.check_call(["git", "clone", "--quiet", "--branch", BRANCH, REPO, "repo"])
        get_ipython().run_line_magic("cd", "repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print(f"Repositorio ({BRANCH}) y dependencias listas en Colab")
else:
    print("Entorno local: dependencias existentes")

Entorno local: dependencias existentes


In [2]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
assert (ROOT / "src").exists(), "Ejecutar desde la raíz del repositorio"
ARTIFACTS = ROOT / "artifacts"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

from src.data.dataset import load_split
from src.evaluation.anomaly import (
    anomaly_metrics,
    error_by_feature_group,
    get_anomaly_score,
    reconstruction_scores,
    select_threshold,
)
from src.models.autoencoder import SequenceAutoencoder, fit_autoencoder, masked_reconstruction_error
from src.models.encoder import EncoderConfig, save_encoder
from src.utils import get_device, set_seed

set_seed(42)
device = get_device()
start = time.perf_counter()
print("Raíz:", ROOT)
print("Dispositivo:", device)

Raíz: C:\Users\richi\Documents\2026_S2_Local\Deep-Learning
Dispositivo: cpu


## 1. Qué datos usa la Etapa A y por qué solo los normales

Lo pensamos como entrenar a un analista mostrándole, durante meses, únicamente
expedientes de clientes normales. Nunca ve un caso de lavado. Cuando después intenta
"recordar de memoria" un expediente nuevo, lo hace bien si se parece a lo que ya
conoce, y mal si es distinto. Ese error al reconstruir es la señal que buscamos.

Por eso, del split de **entrenamiento** usamos exclusivamente el subconjunto de
remitentes normales (`y == 0`). Las secuencias positivas de TRAIN no participan en
absoluto. De **validación** separamos dos usos distintos y no intercambiables:

- los remitentes normales de VAL nos sirven para el *early stopping*, porque comparten
  la distribución del entrenamiento y no queremos decidir cuándo parar con datos que el
  modelo jamás debería ver;
- los remitentes positivos de VAL los reservamos exclusivamente para elegir el umbral
  de anomalía más adelante. Es la primera vez que el modelo "ve" indirectamente una
  etiqueta, y solo para calibrar, nunca para entrenar pesos.

In [3]:
splits = {name: load_split(ARTIFACTS / f"{name}.npz") for name in ("train", "val", "test")}
feature_names = json.loads((ARTIFACTS / "feature_names.json").read_text(encoding="utf-8"))
num_features = len(feature_names)

train, val, test = splits["train"], splits["val"], splits["test"]

train_normal_idx = np.flatnonzero(train["y"] == 0)
val_normal_idx = np.flatnonzero(val["y"] == 0)
val_positive_idx = np.flatnonzero(val["y"] == 1)

assert int(train["y"][train_normal_idx].sum()) == 0, "Se coló una secuencia positiva al entrenamiento"

X_train_normal = train["X"][train_normal_idx]
lengths_train_normal = train["lengths"][train_normal_idx].astype(np.int64)
mask_train_normal = train["mask"][train_normal_idx]

X_val_normal = val["X"][val_normal_idx]
lengths_val_normal = val["lengths"][val_normal_idx].astype(np.int64)
mask_val_normal = val["mask"][val_normal_idx]

counts = pd.DataFrame(
    {
        "subconjunto": [
            "TRAIN normal (entrenamiento)",
            "VAL normal (early stopping)",
            "VAL positivo (solo umbral)",
        ],
        "secuencias": [len(train_normal_idx), len(val_normal_idx), len(val_positive_idx)],
    }
)
display(counts)

,subconjunto,secuencias
0,TRAIN normal (entrenamiento),38620
1,VAL normal (early stopping),8260
2,VAL positivo (solo umbral),413


**Lectura:** confirmamos que el entrenamiento usa únicamente las secuencias normales de
TRAIN, que los normales de VAL calibran cuándo detenerlo, y que los positivos de VAL
quedan aparte para elegir el umbral más adelante, nunca para ajustar un solo peso del
autoencoder.

## 2. Arquitectura y justificación

Con los datos ya separados, toca definir el modelo que va a aprender la normalidad.

| Componente | Elección | Justificación |
|---|---|---|
| Encoder | GRU 1 capa, `hidden_size=64`, con `pack_padded_sequence` | Maneja longitud variable de forma nativa. Preferimos GRU sobre LSTM por tener menos parámetros: con solo 1,931 positivos en todo el dataset nos conviene un modelo compacto |
| Bottleneck | Lineal `64 → 32` | Comprime 32×49 = 1,568 números a 32, una razón aproximada de 49 a 1, y fuerza al modelo a quedarse solo con lo esencial del comportamiento |
| Decoder | Repetir el latente en los `T` pasos + GRU + lineal `64 → 49` | Lo dejamos sin autorregresión para evitar *exposure bias* y teacher forcing. Es determinista, más rápido, y es el patrón estándar de autoencoder recurrente para detección de anomalías |

**Por qué estas arquitecturas y no otras de las que vimos.** La GRU viene de la Semana 3, y
la preferimos sobre la LSTM porque usa tres compuertas en vez de cuatro, o sea menos
parámetros: con 1,931 positivos en todo el dataset, cada parámetro de más es una
oportunidad de sobreajuste. Descartamos la convolución de la Semana 2 porque impone
invarianza a la traslación, es decir asume que un patrón significa lo mismo esté donde
esté, y en lavado no funciona así: una ráfaga de transferencias al final del historial no
equivale a la misma ráfaga al principio. Y descartamos los Transformers de la Semana 6
porque rinden con secuencias largas y mucho dato, mientras que las nuestras tienen mediana
15 y el conjunto de positivos es pequeño.

**Contrato no negociable con la Etapa B:** hicimos que el encoder devuelva, además del
vector latente, los **estados ocultos por timestep** `[B, 32, 64]`. Ese tensor es la
entrada de la atención de la Etapa B. Sin él no habría heatmap ni explicación por
transacción en el MVP.

In [4]:
config = EncoderConfig(num_features=num_features, hidden_size=64, latent_size=32, num_layers=1, dropout=0.0)
model = SequenceAutoencoder(config)

dummy_x = torch.from_numpy(X_train_normal[:4])
dummy_lengths = torch.from_numpy(lengths_train_normal[:4])
dummy_x_hat, dummy_hidden, dummy_latent = model(dummy_x, dummy_lengths)

num_params = sum(p.numel() for p in model.parameters())
print(f"hidden_states (entrada de atención Etapa B): {tuple(dummy_hidden.shape)}")
print(f"latent (vector comprimido):                  {tuple(dummy_latent.shape)}")
print(f"reconstrucción:                              {tuple(dummy_x_hat.shape)}")
print(f"Parámetros totales: {num_params:,}")

hidden_states (entrada de atención Etapa B): (4, 32, 64)
latent (vector comprimido):                  (4, 32)
reconstrucción:                              (4, 32, 49)
Parámetros totales: 46,161


**Verificación:** `hidden_states` tiene forma `[batch, 32, 64]`, es decir un vector por
cada una de las 32 transacciones y no solo un resumen final. Gracias a esta comprobación
sabemos que el contrato con la Etapa B se cumple antes de invertir tiempo en entrenar.

## 3. Pérdida enmascarada

Con la arquitectura ya verificada, el siguiente problema es cómo medir el error de
reconstrucción sin engañarnos.

Calculamos la pérdida como MSE dividido entre `pasos_reales × num_features`, y no entre
`batch × 32 × num_features`. Con post-padding a 32, una secuencia de longitud 4 tiene 28
pasos en cero que cualquier decoder reconstruye trivialmente. Sin máscara, el error
promedio de una secuencia corta bajaría solo por ser corta y no por ser normal, y nuestro
score terminaría midiendo *longitud de historial* en vez de *rareza de comportamiento*.

En la celda siguiente medimos ese efecto en lugar de darlo por supuesto.

Es un caso concreto de lo que trabajamos en la Semana 8, Funciones de Pérdida,
Regularización y Optimización: la función de pérdida no se toma de catálogo, se adapta a la
estructura del dato. Aquí el MSE estándar habría medido lo que no queríamos.

In [5]:
with torch.no_grad():
    x_val_all = torch.from_numpy(val["X"])
    lengths_val_all = torch.from_numpy(val["lengths"].astype(np.int64))
    mask_val_all = torch.from_numpy(val["mask"])
    x_hat_untrained, _, _ = model(x_val_all, lengths_val_all)

masked_error = masked_reconstruction_error(
    x_val_all, x_hat_untrained, mask_val_all, per_sequence=True
).numpy()
full_mask = torch.ones_like(mask_val_all)
unmasked_error = masked_reconstruction_error(
    x_val_all, x_hat_untrained, full_mask, per_sequence=True
).numpy()

corr_unmasked = float(np.corrcoef(val["lengths"], unmasked_error)[0, 1])
corr_masked = float(np.corrcoef(val["lengths"], masked_error)[0, 1])
print(f"Correlación longitud↔error SIN máscara (modelo sin entrenar): {corr_unmasked:+.3f}")
print(f"Correlación longitud↔error CON máscara (modelo sin entrenar): {corr_masked:+.3f}")

Correlación longitud↔error SIN máscara (modelo sin entrenar): +0.556
Correlación longitud↔error CON máscara (modelo sin entrenar): -0.149


**Lectura del experimento:** comprobamos que sin máscara la correlación entre longitud y
error es fuertemente positiva, +0.556. Un modelo sin entrenar ya "parece" detectar
anomalías simplemente porque las secuencias cortas tienen más padding y se reconstruyen
trivialmente. Con máscara esa correlación cae a -0.149, o sea que el efecto casi
desaparece, y todo esto lo medimos *antes de entrenar nada*.

Gracias a este experimento confirmamos que la máscara es indispensable y no un detalle
menor. Nos resulta especialmente relevante porque las secuencias sospechosas son en
promedio más cortas que las normales en TRAIN, 13.48 contra 16.03 transacciones: sin
máscara, el score confundiría "historial corto" con "comportamiento normal" justo en la
dirección que inflaría artificialmente nuestro desempeño.

## 4. Entrenamiento

Con la pérdida definida y el riesgo de la máscara ya descartado, pasamos a entrenar.

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| Optimizador | Adam, `lr=1e-3` | Estándar para RNN pequeñas, con una tasa conservadora dado el tamaño del dataset |
| Batch size | 256 | Suficientemente grande para estabilizar el gradiente sobre 38,620 secuencias sin saturar la memoria disponible |
| Épocas máximas | 90 | Con early stopping, lo tratamos como un techo cómodo y no como un objetivo |
| *Early stopping* | Paciencia 10 sobre la pérdida de los normales de VAL | Detiene el entrenamiento cuando deja de mejorar en datos no vistos de la misma distribución |
| `clip_grad` | 1.0 | Estabiliza el entrenamiento de la GRU ante gradientes explosivos |
| Semilla | 42 | Reproducibilidad, consistente con el resto del proyecto |

Los cuatro elementos de esta tabla que no son arquitectura vienen de la Semana 8: Adam como
optimizador, el recorte de gradiente para que la GRU no explote, el early stopping como
regularización implícita, y la semilla fija para que el resultado sea reproducible.

In [6]:
train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_train_normal),
        torch.from_numpy(lengths_train_normal),
        torch.from_numpy(mask_train_normal),
    ),
    batch_size=256,
    shuffle=True,
)
val_normal_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_val_normal),
        torch.from_numpy(lengths_val_normal),
        torch.from_numpy(mask_val_normal),
    ),
    batch_size=256,
)

set_seed(42)
model = SequenceAutoencoder(config)
train_start = time.perf_counter()
history = fit_autoencoder(
    model,
    train_loader,
    val_normal_loader,
    epochs=90,
    lr=1e-3,
    patience=10,
    device=device,
    clip_grad=1.0,
)
train_minutes = (time.perf_counter() - train_start) / 60
print(f"Épocas ejecutadas: {history['epochs_run']} (mejor época: {history['best_epoch']})")
print(f"Pérdida de validación (normales) en la mejor época: {history['best_val_loss']:.5f}")
print(f"Tiempo de entrenamiento: {train_minutes:.2f} minutos")

Épocas ejecutadas: 90 (mejor época: 89)
Pérdida de validación (normales) en la mejor época: 0.06084
Tiempo de entrenamiento: 27.63 minutos


In [7]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["train_loss"], label="Train (normal)")
ax.plot(history["val_loss"], label="Val (normal)")
ax.axvline(history["best_epoch"], color="grey", linestyle="--", label="Mejor época")
ax.set_xlabel("Época")
ax.set_ylabel("MSE enmascarada")
ax.set_title("Curva de entrenamiento — autoencoder de normalidad")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "stage_a_training_curve.png", dpi=110)
plt.show()

C:\Users\richi\AppData\Local\Temp\ipykernel_20284\1755026395.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura de las curvas:** el entrenamiento corrió las 90 épocas completas sin que el
*early stopping* (paciencia 10) llegara a activarse. La pérdida de validación seguía
bajando, aunque de forma cada vez más marginal. No vemos señal de sobreajuste, porque
train y val se mueven juntas, pero tampoco una convergencia clara a un mínimo estable.

**Bitácora de nuestras pruebas.** Durante el desarrollo entrenamos este modelo tres
veces, con 60, 150 y 90 épocas, para decidir dónde poner el techo. Con 60 épocas
obtuvimos ROC-AUC 0.756 y PR-AUC 0.284 en validación; con 90 bajamos a 0.751 y 0.272.
Es decir, entrenar más redujo el error de reconstrucción pero empeoró ligeramente la
capacidad de separar normal de sospechoso. No dejamos esas corridas dentro del notebook
para no triplicar su tiempo de ejecución, pero el hallazgo es el que explica por qué nos
quedamos con este checkpoint: minimizar el error de reconstrucción no es nuestro
objetivo real, separar comportamiento normal de anómalo sí lo es.

## 5. Del error de reconstrucción al score de anomalía

Ya con el modelo entrenado, lo convertimos en un detector.

El score de anomalía de una secuencia es su error de reconstrucción enmascarado
(`masked_reconstruction_error(..., per_sequence=True)`): qué tan mal logra el autoencoder,
que solo conoce normalidad, reproducir esa secuencia concreta. Lo calculamos sobre todo
VALIDATION, normales y positivos, para poder comparar las dos distribuciones.

In [8]:
val_scores = reconstruction_scores(model, val["X"], val["mask"], val["lengths"], device=device)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(val_scores[val["y"] == 0], bins=40, alpha=0.6, density=True, label="Normal")
ax.hist(val_scores[val["y"] == 1], bins=40, alpha=0.6, density=True, label="Sospechoso")
ax.set_xlabel("Error de reconstrucción (score de anomalía)")
ax.set_ylabel("Densidad")
ax.set_title("Distribución del score de anomalía — VALIDATION")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "stage_a_score_distribution.png", dpi=110)
plt.show()

val_metrics = anomaly_metrics(val_scores, val["y"])
print(json.dumps(val_metrics, indent=2))

{
  "roc_auc": 0.7507460323974462,
  "pr_auc": 0.2719037809755941,
  "n_sequences": 8673,
  "n_positive": 413
}


C:\Users\richi\AppData\Local\Temp\ipykernel_20284\25523656.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura:** obtuvimos ROC-AUC 0.751 y PR-AUC 0.272 sobre 4.76% de positivos, muy por
encima del azar (0.048) aunque lejos de una separación perfecta. El histograma muestra
solapamiento visible entre las dos distribuciones: hay remitentes normales con error alto
y sospechosos con error bajo.

Con esto pudimos ver que un autoencoder no supervisado sobre este problema separa de forma
parcial. Por eso en la Etapa B le sumamos aprendizaje supervisado en lugar de depender solo
de esta señal.

In [9]:
# El modelo quedó en `device` al entrenar, así que los tensores tienen que ir con él.
with torch.no_grad():
    x_val_hat, _, _ = model(x_val_all.to(device), lengths_val_all)

group_errors = error_by_feature_group(
    val["X"], x_val_hat.cpu().numpy(), val["mask"], feature_names
)
display(pd.DataFrame([group_errors]).T.rename(columns={0: "MSE promedio (VAL)"}))

,MSE promedio (VAL)
continuous,0.294411
binary,0.053175
one_hot,0.016867


**Decisión sobre la pérdida:** de las 49 features, el grupo continuo (8 columnas: montos,
gap temporal y componentes cíclicas) tiene el MSE promedio más alto, 0.294, muy por
encima del grupo binario (0.053) y del one-hot (0.017), a pesar de tener muchas menos
dimensiones que este último, 8 contra 37.

En la práctica ocurrió lo contrario de lo que temíamos: las 37 columnas one-hot no ahogan
a las continuas, porque son fáciles de reconstruir (la moneda y el formato de pago son
casi constantes por remitente) y su error cae rápido durante el entrenamiento. El grupo
continuo, donde vive la señal de montos fragmentados propia del lavado, es el que domina
la pérdida total.

Gracias a esta descomposición decidimos mantener la MSE plana sin ponderar por grupo. No
hace falta, y añadir pesos manuales sería una complejidad injustificada dado el resultado.

In [10]:
train_normal_scores = reconstruction_scores(
    model, X_train_normal, mask_train_normal, lengths_train_normal, device=device
)

candidates = [
    select_threshold(val_scores, val["y"], method="f1_max"),
    select_threshold(val_scores, val["y"], method="budget", budget=0.05),
    select_threshold(
        val_scores,
        val["y"],
        method="percentile",
        reference_scores=train_normal_scores,
        percentile=95,
    ),
]
display(pd.DataFrame(candidates)[["method", "threshold", "precision", "recall", "f1", "alert_rate"]])

,method,threshold,precision,recall,f1,alert_rate
0,f1_max,0.117777,0.455263,0.418886,0.436318,0.043814
1,budget,0.106232,0.414747,0.435835,0.425030,0.050040
2,percentile,0.083072,0.326531,0.503632,0.396190,0.073446


**Justificación del umbral:** elegimos el de **F1 máximo** (`method="f1_max"`, valor
0.1178). Con 4.76% de positivos, la exactitud no informa y ROC-AUC resulta optimista frente
al desbalance, mientras que F1 (0.436) pondera directamente precisión (45.5%) y recall
(41.9%) sobre la clase que nos interesa. Además fue el mejor de los tres candidatos que
evaluamos.

El de presupuesto fijo (`budget=5%`, F1=0.425) quedó muy cerca y tiene una lectura
operativa clara: revisar solo el 5% de mayor score. El percentil 95 de TRAIN (F1=0.396) lo
dejamos como referencia clásica de detección de anomalías, con mayor recall (50.4%) pero
mucha menor precisión (32.7%), o sea más alertas y más ruido.

En términos de carga de trabajo: con el umbral que elegimos, de cada 1,000 remitentes
generamos unas **44 alertas** (4.38%), y de ellas **45.5%** son positivos reales. Un equipo
de cumplimiento que hoy revisa alertas de un sistema de reglas trabaja con tasas de acierto
de 1 a 5%, así que 45.5% ya es una mejora grande, aunque todavía significa que más de la
mitad del trabajo se va en falsas alarmas. Es la señal que en la Etapa B combinamos con
aprendizaje supervisado.

In [11]:
final_corr = float(np.corrcoef(val["lengths"], val_scores)[0, 1])
print(f"Correlación longitud↔score (modelo entrenado): {final_corr:+.3f}")

from sklearn.metrics import average_precision_score, roc_auc_score

baseline_length = -val["lengths"].astype(np.float64)
baseline_amount = val["X"][:, :, 0].mean(axis=1)  # log_amount_paid promedio

baselines = pd.DataFrame(
    [
        {
            "baseline": "autoencoder (Etapa A)",
            "roc_auc": val_metrics["roc_auc"],
            "pr_auc": val_metrics["pr_auc"],
        },
        {
            "baseline": "longitud de secuencia (trivial)",
            "roc_auc": roc_auc_score(val["y"], baseline_length),
            "pr_auc": average_precision_score(val["y"], baseline_length),
        },
        {
            "baseline": "monto promedio (trivial)",
            "roc_auc": roc_auc_score(val["y"], baseline_amount),
            "pr_auc": average_precision_score(val["y"], baseline_amount),
        },
    ]
)
display(baselines)

Correlación longitud↔score (modelo entrenado): +0.230


,baseline,roc_auc,pr_auc
0,autoencoder (Etapa A),0.750746,0.271904
1,longitud de secuencia (trivial),0.551685,0.054776
2,monto promedio (trivial),0.572181,0.052671


**Qué logra y qué no logra la Etapa A:** comprobamos que el score del autoencoder supera
ampliamente a los baselines triviales de longitud y monto promedio, con PR-AUC 0.272 contra
0.055 y 0.053 respectivamente, casi 5 veces más. Con este resultado sabemos que aprende
señal propia y no solo redescubre el confound de longitud que detectamos en el experimento
de la máscara.

La correlación entre longitud y score del modelo entrenado (+0.230) es bastante menor que
la del modelo sin entrenar y sin máscara (+0.556). Como las secuencias sospechosas son más
cortas en promedio, una correlación positiva trabaja *en contra* de una detección
artificialmente buena por ese atajo, así que la leemos como un resultado tranquilizador.

Limitación: según la bitácora de pruebas que describimos más arriba, entrenar más allá de
60 épocas siguió bajando el error de reconstrucción pero redujo levemente la separación
entre normal y sospechoso. Es un efecto conocido en autoencoders para detección de
anomalías, donde un modelo demasiado bien entrenado empieza a reconstruir también las
anomalías. Minimizar el error de reconstrucción y separar bien las clases son dos objetivos
distintos, y por eso la ablación de la Etapa B es la que debe mostrar cuánto aporta de
verdad esta señal.

## 6. Evaluación final en TEST

Con el umbral elegido y los controles de honestidad hechos, queda la prueba final.

Tocamos TEST una sola vez, con el umbral ya congelado a partir de VALIDATION. No
volvemos a ajustar nada después de ver este resultado.

In [12]:
test_scores = reconstruction_scores(model, test["X"], test["mask"], test["lengths"], device=device)

chosen_candidate = candidates[0]  # f1_max, justificado en la sección 5
chosen_threshold = chosen_candidate["threshold"]

test_metrics = anomaly_metrics(test_scores, test["y"])
predicted_test = (test_scores >= chosen_threshold).astype(int)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test["y"], predicted_test)
print(f"Umbral aplicado (elegido en VAL, método={chosen_candidate['method']}): {chosen_threshold:.5f}")
print(json.dumps(test_metrics, indent=2))
display(
    pd.DataFrame(
        cm,
        index=["real_normal", "real_sospechoso"],
        columns=["pred_normal", "pred_sospechoso"],
    )
)

Umbral aplicado (elegido en VAL, método=f1_max): 0.11778
{
  "roc_auc": 0.7670934560899905,
  "pr_auc": 0.30842159511867717,
  "n_sequences": 8694,
  "n_positive": 414
}


,pred_normal,pred_sospechoso
real_normal,8074,206
real_sospechoso,222,192


## 7. Artefactos y contrato para la Etapa B

Para cerrar, guardamos lo que la Etapa B va a necesitar y verificamos que se recargue igual desde disco.

```python
from src.models.encoder import load_encoder
from src.evaluation.anomaly import get_anomaly_score

encoder, config = load_encoder("artifacts/encoder.pt")
hidden_states, latent = encoder(x, lengths)   # [B, 32, 64] y [B, 32]
```

`hidden_states` es la entrada de la atención de la Etapa B. `latent` puede usarse como
representación comprimida adicional para la cabeza clasificadora. `get_anomaly_score`
alimenta la combinación de señales que evaluamos en la Etapa B.

**Para que nuestro experimento de ablación de la Etapa B sea válido**, hay que instanciar
`SequenceEncoder` con la misma `EncoderConfig` en ambos brazos, el preentrenado y el de
inicialización aleatoria. Por eso dejamos la clase en `src/models/encoder.py` y no la
duplicamos.

Guardar el encoder por separado del autoencoder completo es lo que habilita la estrategia
de la Semana 9, Transfer Learning y Fine-Tuning. Sin un `encoder.pt` independiente, la
Etapa B tendría que empezar de cero y el proyecto perdería su premisa.

In [13]:
threshold_report = {
    "config": {
        "num_features": config.num_features,
        "hidden_size": config.hidden_size,
        "latent_size": config.latent_size,
        "num_layers": config.num_layers,
        "dropout": config.dropout,
    },
    "normal_train_error_stats": {
        "mean": float(train_normal_scores.mean()),
        "std": float(train_normal_scores.std()),
    },
    "candidates": {c["method"]: c for c in candidates},
    "selected": {"method": chosen_candidate["method"], "value": float(chosen_threshold)},
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "feature_error_by_group": group_errors,
    "honesty_checks": {
        "length_score_correlation": final_corr,
        "baselines": baselines.to_dict(orient="records"),
    },
}
(ARTIFACTS / "anomaly_threshold.json").write_text(
    json.dumps(threshold_report, indent=2, ensure_ascii=False), encoding="utf-8"
)
torch.save(
    {"state_dict": model.state_dict(), "config": threshold_report["config"]},
    ARTIFACTS / "stage_a_model.pt",
)
save_encoder(ARTIFACTS / "encoder.pt", model.encoder, config)
print("Artefactos guardados en", ARTIFACTS)

# Round-trip: recargar todo desde disco y confirmar que el score no cambia
from src.evaluation import anomaly as anomaly_module

anomaly_module._STAGE_A_CACHE.clear()

sample_idx = 0
sample_result = get_anomaly_score(
    test["X"][sample_idx],
    int(test["lengths"][sample_idx]),
    model_path=ARTIFACTS / "stage_a_model.pt",
    threshold_path=ARTIFACTS / "anomaly_threshold.json",
)
print("\nDemo get_anomaly_score sobre un remitente de TEST:")
print(json.dumps(sample_result, indent=2))
round_trip_ok = abs(sample_result["score"] - float(test_scores[sample_idx])) < 1e-4
print(f"\nScore calculado en la celda de evaluación TEST: {test_scores[sample_idx]:.5f}")
print(f"Coincide con el round-trip desde disco: {round_trip_ok}")
assert round_trip_ok, "El score recargado desde disco no coincide, revisar guardado/carga"

print(f"\nTiempo total del notebook: {(time.perf_counter() - start) / 60:.2f} minutos")

Artefactos guardados en C:\Users\richi\Documents\2026_S2_Local\Deep-Learning\artifacts

Demo get_anomaly_score sobre un remitente de TEST:
{
  "score": 0.04857586696743965,
  "z_score": 0.12606783856246875,
  "threshold": 0.11777700483798981,
  "is_anomalous": false
}

Score calculado en la celda de evaluación TEST: 0.04858
Coincide con el round-trip desde disco: True

Tiempo total del notebook: 27.79 minutos


## Cierre

**Limitaciones de esta etapa:**

- Nuestro autoencoder no supervisado logra una separación parcial (ROC-AUC 0.751 y
  PR-AUC 0.272 en VALIDATION; 0.767 y 0.308 en TEST), consistente con lo que reporta la
  literatura: la normalidad de remesas es heterogénea y no todo lo "raro" es lavado.
- El entrenamiento no convergió a un mínimo estable dentro del presupuesto de 90 épocas,
  porque la pérdida de validación seguía bajando de forma marginal. Aun así, según
  nuestra bitácora de pruebas, entrenar más empeoró ligeramente la separación entre
  normal y sospechoso, así que extender el entrenamiento no era el camino para mejorar
  este componente.
- Calibramos el umbral sobre un dataset submuestreado 20:1 (normal a positivo), que no
  representa la prevalencia operativa real de 0.102% por transacción. Habría que
  recalibrarlo antes de llevarlo a producción.
- Una ventana por remitente, heredada de `01_data_engineering.ipynb`, limita el historial
  visible para las cuentas muy activas.

**Qué sigue:** en la Etapa B (`03_stage_b_classifier.ipynb`) cargamos
`artifacts/encoder.pt`, le agregamos atención y una cabeza clasificadora supervisada, y
demostramos con un experimento de ablación si partir de este encoder preentrenado aporta
valor sobre entrenar un clasificador desde cero.